In [1]:
import importlib.util
import json
import logging
import sys
import time
from datetime import datetime, timezone
from io import StringIO
from pathlib import Path
from time import perf_counter

required_modules = ["joblib", "pandas", "sklearn"]
missing_modules = [
    name for name in required_modules
    if importlib.util.find_spec(name) is None
]
if missing_modules:
    raise RuntimeError(
        "Missing project libraries: " + ", ".join(missing_modules)
        + ". Activate the project environment; this notebook will not install packages."
    )

import joblib
import pandas as pd


def find_project_root(start=Path.cwd()):
    for folder in (start, *start.parents):
        if (folder / "PROBLEM.md").is_file() and (folder / "models" / "model.joblib").is_file():
            return folder
    raise FileNotFoundError(
        "Could not locate the MP202 project root containing PROBLEM.md and models/model.joblib."
    )


PROJECT_ROOT = find_project_root()
MODEL_PATH = PROJECT_ROOT / "models" / "model.joblib"
model = joblib.load(MODEL_PATH)

print("Python       :", sys.version.split()[0])
print("Project root :", PROJECT_ROOT)
print("Model        :", MODEL_PATH.relative_to(PROJECT_ROOT))
print("Model loaded. Continue to Step 1.")

Python       : 3.14.4
Project root : /home/ak/G205_MP202
Model        : models/model.joblib
Model loaded. Continue to Step 1.


In [2]:
class MP202JsonFormatter(logging.Formatter):
    """Format an allowlisted set of MP202 service fields as one JSON line."""

    def format(self, record):
        event = {
            "timestamp": datetime.fromtimestamp(
                record.created, tz=timezone.utc
            ).isoformat(timespec="milliseconds"),
            "level": record.levelname,
            "service": "mp202-api",
            "event": getattr(record, "event_name", "api_event"),
            "status": getattr(record, "status", "unknown"),
            "route": getattr(record, "route", "/predict"),
            "request_id": getattr(record, "request_id", None),
            "duration_ms": getattr(record, "duration_ms", None),
        }
        return json.dumps(event, separators=(",", ":"), allow_nan=False)


logger = logging.getLogger("mp202.api")
logger.setLevel(logging.DEBUG)
logger.propagate = False
logger.handlers.clear()

json_formatter = MP202JsonFormatter()
stdout_handler = logging.StreamHandler(sys.stdout)
stdout_handler.setFormatter(json_formatter)
memory_buffer = StringIO()
memory_handler = logging.StreamHandler(memory_buffer)
memory_handler.setFormatter(json_formatter)
logger.addHandler(stdout_handler)
logger.addHandler(memory_handler)


def log_event(level, event_name, *, status, request_id=None, duration_ms=None):
    """Write one structured event without logging request or customer details."""
    logger.log(
        level,
        event_name,
        extra={
            "event_name": event_name,
            "status": status,
            "route": "/predict",
            "request_id": request_id,
            "duration_ms": duration_ms,
        },
    )


print("Logger ready: JSON goes to stdout; search capture is in memory only.")

Logger ready: JSON goes to stdout; search capture is in memory only.


In [3]:
log_event(logging.DEBUG, "debug_example", status="example")
log_event(logging.INFO, "service_ready", status="ok")
log_event(logging.WARNING, "latency_warning", status="slow")
log_event(logging.ERROR, "prediction_failed", status="error")
log_event(logging.CRITICAL, "service_unavailable", status="critical")

{"timestamp":"2026-11-27T07:38:11.540+00:00","level":"DEBUG","service":"mp202-api","event":"debug_example","status":"example","route":"/predict","request_id":null,"duration_ms":null}
{"timestamp":"2026-11-27T07:38:11.543+00:00","level":"INFO","service":"mp202-api","event":"service_ready","status":"ok","route":"/predict","request_id":null,"duration_ms":null}
{"timestamp":"2026-11-27T07:38:11.544+00:00","level":"WARNING","service":"mp202-api","event":"latency_warning","status":"slow","route":"/predict","request_id":null,"duration_ms":null}
{"timestamp":"2026-11-27T07:38:11.545+00:00","level":"ERROR","service":"mp202-api","event":"prediction_failed","status":"error","route":"/predict","request_id":null,"duration_ms":null}
{"timestamp":"2026-11-27T07:38:11.547+00:00","level":"CRITICAL","service":"mp202-api","event":"service_unavailable","status":"critical","route":"/predict","request_id":null,"duration_ms":null}


In [4]:
LEVELS = {
    "DEBUG": logging.DEBUG,
    "INFO": logging.INFO,
    "WARNING": logging.WARNING,
    "ERROR": logging.ERROR,
    "CRITICAL": logging.CRITICAL,
}

logger.setLevel(logging.WARNING)
T1_kept = [name for name, level in LEVELS.items() if logger.isEnabledFor(level)]
print("Recorded at WARNING threshold:", T1_kept)
logger.setLevel(logging.INFO)

Recorded at WARNING threshold: ['WARNING', 'ERROR', 'CRITICAL']


In [5]:
sample_record = logger.makeRecord(
    name="mp202.api",
    level=logging.INFO,
    fn="notebook",
    lno=1,
    msg="prediction_served",
    args=(),
    exc_info=None,
    extra={
        "event_name": "prediction_served",
        "status": "success",
        "route": "/predict",
        "request_id": "demo-0001",
        "duration_ms": 4.25,
        "brand_tier": "not included",
    },
)
T2_line = json.loads(MP202JsonFormatter().format(sample_record))
print(json.dumps(T2_line, indent=2))

{
  "timestamp": "2026-11-27T07:38:21.393+00:00",
  "level": "INFO",
  "service": "mp202-api",
  "event": "prediction_served",
  "status": "success",
  "route": "/predict",
  "request_id": "demo-0001",
  "duration_ms": 4.25
}


In [7]:
FEATURE_COLUMNS = [
    "age_years",
    "km_driven",
    "engine_cc",
    "owners",
    "service_records",
    "accidents",
    "brand_tier",
    "city_zone",
]
BRANDS = ["budget", "mid", "premium"]
ZONES = ["North", "South", "Central"]
SIMULATED_FAILURES = {11, 27}
SIMULATED_SLOW_CALLS = {4, 17, 31}


def synthetic_bike(index):
    return {
        "age_years": 1 + index % 8,
        "km_driven": 8000 + index * 500,
        "engine_cc": 100 + (index % 8) * 50,
        "owners": 1 + index % 3,
        "service_records": index % 10,
        "accidents": index % 2,
        "brand_tier": BRANDS[index % len(BRANDS)],
        "city_zone": ZONES[index % len(ZONES)],
    }


def predict_and_log(bike, request_id, *, simulate_failure=False, simulate_slow=False):
    started = perf_counter()

    if simulate_failure:
        log_event(
            logging.ERROR,
            "prediction_failed",
            status="error",
            request_id=request_id,
            duration_ms=round((perf_counter() - started) * 1000, 2),
        )
        return None

    if simulate_slow:
        time.sleep(0.12)

    features = pd.DataFrame([bike], columns=FEATURE_COLUMNS)
    prediction = float(model.predict(features)[0])
    duration_ms = round((perf_counter() - started) * 1000, 2)
    log_event(
        logging.INFO,
        "prediction_served",
        status="success",
        request_id=request_id,
        duration_ms=duration_ms,
    )
    return prediction


prediction_results = []
for index in range(40):
    request_id = f"demo-{index + 1:04d}"
    result = predict_and_log(
        synthetic_bike(index),
        request_id,
        simulate_failure=index in SIMULATED_FAILURES,
        simulate_slow=index in SIMULATED_SLOW_CALLS,
    )
    prediction_results.append((request_id, result))

memory_handler.flush()
events = [
    json.loads(line)
    for line in memory_buffer.getvalue().splitlines()
    if line.strip()
]
print(f"\nCaptured {len(events)} prediction events in memory; no log file was created.")

{"timestamp":"2026-11-27T07:38:44.950+00:00","level":"INFO","service":"mp202-api","event":"prediction_served","status":"success","route":"/predict","request_id":"demo-0001","duration_ms":14.46}
{"timestamp":"2026-11-27T07:38:44.962+00:00","level":"INFO","service":"mp202-api","event":"prediction_served","status":"success","route":"/predict","request_id":"demo-0002","duration_ms":9.9}
{"timestamp":"2026-11-27T07:38:44.973+00:00","level":"INFO","service":"mp202-api","event":"prediction_served","status":"success","route":"/predict","request_id":"demo-0003","duration_ms":9.12}
{"timestamp":"2026-11-27T07:38:44.982+00:00","level":"INFO","service":"mp202-api","event":"prediction_served","status":"success","route":"/predict","request_id":"demo-0004","duration_ms":8.07}


{"timestamp":"2026-11-27T07:38:45.112+00:00","level":"INFO","service":"mp202-api","event":"prediction_served","status":"success","route":"/predict","request_id":"demo-0005","duration_ms":128.51}
{"timestamp":"2026-11-27T07:38:45.122+00:00","level":"INFO","service":"mp202-api","event":"prediction_served","status":"success","route":"/predict","request_id":"demo-0006","duration_ms":8.64}
{"timestamp":"2026-11-27T07:38:45.131+00:00","level":"INFO","service":"mp202-api","event":"prediction_served","status":"success","route":"/predict","request_id":"demo-0007","duration_ms":7.88}
{"timestamp":"2026-11-27T07:38:45.143+00:00","level":"INFO","service":"mp202-api","event":"prediction_served","status":"success","route":"/predict","request_id":"demo-0008","duration_ms":9.46}
{"timestamp":"2026-11-27T07:38:45.155+00:00","level":"INFO","service":"mp202-api","event":"prediction_served","status":"success","route":"/predict","request_id":"demo-0009","duration_ms":10.9}
{"timestamp":"2026-11-27T07:38:45

In [8]:
served_events = [
    event for event in events
    if event["event"] == "prediction_served"
]
error_events = [
    event for event in events
    if event["level"] == "ERROR"
]
slow_events = [
    event for event in served_events
    if event["duration_ms"] > 100
]

print("events       :", len(events))
print("predictions  :", len(served_events))
print("errors       :", len(error_events))
print("over 100 ms  :", len(slow_events))
print(
    "slow fraction :",
    round(len(slow_events) / len(served_events), 3) if served_events else 0,
)
if served_events:
    slowest = max(served_events, key=lambda event: event["duration_ms"])
    print("slowest request:", slowest["request_id"], slowest["duration_ms"], "ms")

events       : 85
predictions  : 76
errors       : 5
over 100 ms  : 6
slow fraction : 0.079
slowest request: demo-0032 133.07 ms


In [9]:
T3_slow_fraction = (
    sum(event["duration_ms"] > 100 for event in served_events) / len(served_events)
    if served_events
    else 0.0
)
T3_worst_id = (
    max(served_events, key=lambda event: event["duration_ms"])["request_id"]
    if served_events
    else None
)

print("fraction over 100 ms:", T3_slow_fraction)
print("slowest request ID  :", T3_worst_id)

fraction over 100 ms: 0.07894736842105263
slowest request ID  : demo-0032


In [10]:
assert "brand_tier" not in T2_line
assert "city_zone" not in T2_line
assert "prediction" not in T2_line
print("PASS: request features and prediction values are excluded from log events.")

PASS: request features and prediction values are excluded from log events.


In [11]:
checks = [
    ("T1 threshold was restored to INFO", logger.level == logging.INFO),
    ("T1 keeps warning and higher levels", set(T1_kept) == {"WARNING", "ERROR", "CRITICAL"}),
    ("T2 has the project service name", T2_line["service"] == "mp202-api"),
    ("T2 is a valid prediction event", T2_line["event"] == "prediction_served" and T2_line["status"] == "success"),
    ("T2 includes request ID and duration", T2_line["request_id"] == "demo-0001" and T2_line["duration_ms"] == 4.25),
    ("40 prediction events were recorded", len(events) == 40),
    ("Two simulated prediction failures were recorded", len(error_events) == 2),
    ("Successful event fields are privacy-safe", all("brand_tier" not in event and "city_zone" not in event and "prediction" not in event for event in events)),
    ("T3 slow fraction is in range", 0 <= T3_slow_fraction <= 1),
    ("T3 worst request is a successful prediction", T3_worst_id in {event["request_id"] for event in served_events}),
]

for label, passed in checks:
    print(f"{'PASS' if passed else 'FAIL'} | {label}")

passed_count = sum(passed for _, passed in checks)
print(f"\n{passed_count} of {len(checks)} checks passed")

PASS | T1 threshold was restored to INFO
PASS | T1 keeps warning and higher levels
PASS | T2 has the project service name
PASS | T2 is a valid prediction event
PASS | T2 includes request ID and duration
FAIL | 40 prediction events were recorded
FAIL | Two simulated prediction failures were recorded
PASS | Successful event fields are privacy-safe
PASS | T3 slow fraction is in range
PASS | T3 worst request is a successful prediction

8 of 10 checks passed
